### Todo

- Implement conv/mlp extractor ontop of baseFeatureExtractor to be used with multiInputPolicy. 
- Need to figure out if lstm can be implemented as an arg. I.e., how does recurrent ppo class work in sb3. 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from dm_control import suite
import gymnasium as gym
from dm_control.suite.wrappers import pixels
from gym import spaces, Env


In [ ]:
from agent.extractors.vision_net import ConvReluNet

ModuleNotFoundError: No module named 'agent'

In [244]:
env = suite.load(domain_name="swimmer", task_name="swimmer6")
env = pixels.Wrapper(env, pixels_only=False, render_kwargs={'height': 64, 'width': 64, 'camera_id': 2})
time_step = env.reset()
print(time_step.observation.keys())

odict_keys(['joints', 'to_target', 'body_velocities', 'pixels'])


In [245]:
print(time_step.observation['pixels'].shape)
print(time_step.observation['joints'].shape)
print(time_step.observation['body_velocities'].shape)

(25, 25, 3)
(5,)
(18,)


In [246]:
propio  = np.concatenate((time_step.observation['joints'], time_step.observation['body_velocities']))
proprio_tensor = torch.tensor(propio, dtype=torch.float32)
FF = FfFeatureExtractor(conv_output_size=64, proprio_input_size=proprio_tensor.shape[0], mlp_arch=[64, 64])
mlp = FF._build_mlp(proprio_tensor.shape[0], [64, 64])

stacked_propio_tensors = torch.stack([proprio_tensor, proprio_tensor, proprio_tensor])
mlp(proprio_tensor).shape

torch.Size([64])

In [247]:
image = time_step.observation['pixels'].copy()
image_tensor = (
    torch.tensor(image, dtype=torch.float32)
    .permute(2, 0, 1)  # Change shape from (H, W, C) to (C, H, W)
    .unsqueeze(0)  # Add batch dimension: (1, C, H, W)
        )

stacked_image_tensors = torch.stack([image_tensor.squeeze(0), image_tensor.squeeze(0), image_tensor.squeeze(0)])
vision_net = ConvReluNet(output_size=288, training=True)
vision_net(image_tensor)

tensor([[  1.0382,   2.5713,  -1.9584,  -5.2245,  -9.3963,  -3.0149,   4.6739,
           0.4777,  10.4885,   6.4055,   0.7196,   6.5469,   3.2305,   6.9077,
          -4.5061,   3.3283,   2.4718,  14.2163,   7.7413,   6.8718,  -3.0863,
           2.3249,   6.1260,  -3.3798,  -4.1158,  -7.4005,  -6.6231, -18.5022,
          -4.3679,   9.1550,  -7.5016,   7.6776,  -5.8123,   0.2530,  -4.4819,
         -12.3932,   2.5923,  12.3630,   0.5549,  -6.7099,  -6.6140,   6.7471,
           4.4856,   1.9882, -16.1015,   9.4849,  -2.0818,  -4.9503,   0.3468,
           8.9076,   3.4634,  11.3429,  11.6652, -11.7097,   6.3701,  -4.0928,
          11.3617,   6.2832,   0.6153,  -8.9778,   3.6135,   4.5203,   1.7109,
           3.6731,   8.8226,  -9.4465,  -5.5623,  -5.7274,  -2.1554,   6.2744,
           3.9740,  -4.8973,   3.4514,   0.0234,  -8.2042,   0.1453,   2.0617,
          -2.0514,   0.7999,   3.2179,  10.4713,  10.6793,   0.9044,  -9.3962,
          -5.4857,  -3.0993,   1.6797, -19.7563,  -0

In [248]:
image = time_step.observation['pixels'].copy()
image.shape

(25, 25, 3)

In [249]:
conv, prop, comb = FF(image_tensor, proprio_tensor)

In [250]:
print(conv.shape, prop.shape, comb.shape)

torch.Size([1, 64]) torch.Size([1, 64]) torch.Size([1, 128])


In [251]:
proprio_size = env.observation_spec()['body_velocities'].shape[0] + env.observation_spec()['joints'].shape[0]
proprio_size

23

In [252]:
observation_space = gym.spaces.Dict({
    "image": gym.spaces.Box(low=0, high=255, shape=(3, 25, 25), dtype=np.float32),  # Example size
    "proprio": gym.spaces.Box(low=-np.inf, high=np.inf, shape=(proprio_size,), dtype=np.float32)
})


In [253]:
for key, subspace in observation_space.spaces.items():
    print(key, subspace)

image Box(0.0, 255.0, (3, 25, 25), float32)
proprio Box(-inf, inf, (23,), float32)


In [254]:
env.observation_spec()['body_velocities']

Array(shape=(18,), dtype=dtype('float64'), name='body_velocities')

In [255]:
multi = MultiModalFeatureExtractor(observation_space=observation_space, conv_output_size=100, proprio_input_size=proprio_size, mlp_arch=[64, 64])
sample_observation = observation_space.sample()
sample_observation = {key: torch.tensor(sample_observation[key], dtype=torch.float32).unsqueeze(0) for key in sample_observation}
multi(sample_observation).shape

torch.Size([1, 164])